GTZAN Music Genre Classification Fix: Chunk-Level Dataset & Song-Safe Split

Objective

Rebuild the PostgreSQL database so audio_track and vw_clean_tracks point at the real 3-second chunked dataset instead of the raw 30-second originals, and fix the train/val/test split so it's assigned per song instead of per chunk.

Goal

By the end of this fix:


✓ processed/ regenerated with real 3-sec chunks (9,981 files, 10 genres)
✓ 2 tables recreated: music_genre and audio_track (with new song_id column)
✓ 1 view recreated: vw_clean_tracks (now exposes song_id)
✓ 9,981 chunks scanned and loaded with metadata
✓ Corrupted and duplicate chunks flagged and excluded (via song_id, not exact filename)
✓ Stratified 70/15/15 train/val/test split assigned per song, inherited by all its chunks
✓ Verified zero leakage — no song's chunks span more than one split
✓ Database ready for CNN_Training.ipynb re-run

Workflow


→ Diagnose: vw_clean_tracks was pointing at genres_original/ (971 rows, 1 per song), not processed/
→ Fix DATA_ROOT / GENRES_DIR paths in preprocess_clean_final.ipynb
→ Re-run chunking cell → 9,981 chunk files generated
→ Remove stale single-number leftover files from earlier failed runs
→ Drop vw_clean_tracks, audio_track, music_genre
→ Recreate music_genre (unchanged)
→ Recreate audio_track with new song_id column
→ Recreate vw_clean_tracks view (song_id added at end of SELECT)
→ Insert 10 genres (unchanged)
→ Scan processed/ and insert 9,981 chunks, each tagged with song_id
→ Flag corrupted/duplicate chunks by song_id instead of exact filename
→ Assign split by song (stratified 70/15/15, seed 42), inherit split to all chunks
→ Verify zero leakage across splits
→ Final verification

Result
9,695 clean chunks (971 unique songs) → 6,759 train / 1,406 val / 1,530 test.



In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# PostgreSQL connectivity
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Audio processing
import librosa
import soundfile as sf
# Hashing for  data integrity
import hashlib
#importing random seeds=42
import random

import warnings
warnings.filterwarnings('ignore')

# Suppress audioread macOS warnings
import logging
logging.getLogger('audioread').setLevel(logging.ERROR)

print('✓ Libraries loaded')

✓ Libraries loaded


In [15]:
# ── Load credentials from .env ─────────────────────────────────────────────
load_dotenv()

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = 'localhost'
DB_PORT     = 5432
DB_NAME     = 'music_genre_db'

assert DB_USER,     'DB_USER not found in .env'
assert DB_PASSWORD, 'DB_PASSWORD not found in .env'

print(f'✓ Credentials loaded: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

✓ Credentials loaded: ingxrodriguez@localhost:5432/music_genre_db


In [16]:
# ── Create SQLAlchemy engine and test connection ────────────────────────
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Test the connection
try:
    with engine.connect() as conn:
        result = conn.execute(text('SELECT current_database(), current_user;'))
        db_info = result.fetchone()
        print(f'✓ Connected to: {db_info[0]} as {db_info[1]}')
except Exception as e:
    print(f'✗ Connection failed: {e}')

✓ Connected to: music_genre_db as ingxrodriguez


In [17]:
# Define the path to the GTZAN dataset 
GTZAN_PATH = Path('../Data_Music/processed')

# Check if the path exist
if GTZAN_PATH.exists():
    print(f'✓ Dataset found at: {GTZAN_PATH.absolute()}')
else:
    print(f'✗ Dataset NOT found at: {GTZAN_PATH.absolute()}')
    print(f'Check that the folder exists in your project')

# List the genres
genres = sorted([d.name for d in GTZAN_PATH.iterdir() if d.is_dir()])
print(f'\nGenres found: {len(genres)}')
for genre in genres:
    print(f'  - {genre}')

✓ Dataset found at: /Users/ingxrodriguez/music-genre-classification/Sql_Workflow/../Data_Music/processed

Genres found: 10
  - blues
  - classical
  - country
  - disco
  - hiphop
  - jazz
  - metal
  - pop
  - reggae
  - rock


In [26]:
# Point GTZAN_PATH to the CHUNKED dataset (3-second clips), not the raw 30-second originals
GTZAN_PATH = Path('../Data_Music/processed')

# Check if the folder actually exists on disk before we try to scan it
if GTZAN_PATH.exists():
    # Print the full resolved path so we can visually confirm it says "processed", not "genres_original"
    print(f'✓ Dataset found at: {GTZAN_PATH.absolute()}')
else:
    # If the folder is missing, warn clearly instead of silently failing later
    print(f'✗ Dataset NOT found at: {GTZAN_PATH.absolute()}')
    print(f'Check that the folder exists in your project')

# List every subfolder inside processed/ — each one should be a genre name
genres = sorted([d.name for d in GTZAN_PATH.iterdir() if d.is_dir()])

# Print how many genre folders we found (should be 10)
print(f'\nGenres found: {len(genres)}')
for genre in genres:
    print(f'  - {genre}')

# Grab the .wav files inside the FIRST genre folder, just as a sample
sample_genre_files = sorted((GTZAN_PATH / genres[0]).glob('*.wav'))

# Print how many files are in that one folder — should be in the hundreds/thousands (chunks), not ~100 (raw songs)
print(f'\nSample files in {genres[0]}/: {len(sample_genre_files)} total')

# Print the first 3 filenames — expect a pattern like "blues.00000.00.wav" (song + chunk index)
print(sample_genre_files[:3])

✓ Dataset found at: /Users/ingxrodriguez/music-genre-classification/Sql_Workflow/../Data_Music/processed

Genres found: 10
  - blues
  - classical
  - country
  - disco
  - hiphop
  - jazz
  - metal
  - pop
  - reggae
  - rock

Sample files in blues/: 1000 total
[PosixPath('../Data_Music/processed/blues/blues.00000.00.wav'), PosixPath('../Data_Music/processed/blues/blues.00000.01.wav'), PosixPath('../Data_Music/processed/blues/blues.00000.02.wav')]


# Create a table  Music Genre

In [22]:
CREATE_GENRE_TABLE = """
CREATE TABLE music_genre (
    genre_id   SERIAL PRIMARY KEY,
    genre_name VARCHAR(20) NOT NULL UNIQUE
);
"""

with engine.begin() as conn:
    conn.execute(text(CREATE_GENRE_TABLE))
    print('✓ Table music_genre created')

✓ Table music_genre created


In [23]:
#create track table 

CREATE_TRACK_TABLE = """
CREATE TABLE audio_track (
    track_id         SERIAL PRIMARY KEY,
    genre_id         INT NOT NULL REFERENCES music_genre(genre_id) ON DELETE CASCADE,
    file_path        TEXT NOT NULL,
    file_name        VARCHAR(64) NOT NULL,
    song_id          VARCHAR(32) NOT NULL,
    duration_sec     NUMERIC,
    sample_rate      INT,
    channels         INT,
    file_size_bytes  BIGINT,
    file_hash        CHAR(32),
    duplicate_flagged BOOLEAN DEFAULT FALSE,
    corrupted_flagged BOOLEAN DEFAULT FALSE,
    split            VARCHAR(5),
    created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

with engine.begin() as conn:
    conn.execute(text(CREATE_TRACK_TABLE))
    print('✓ Table audio_track created (with song_id column)')

✓ Table audio_track created (with song_id column)


In [ ]:
#create view table for clean tracks

CREATE_VIEW = """
CREATE OR REPLACE VIEW vw_clean_tracks AS
    SELECT 
        at.track_id,
        at.file_path,
        mg.genre_name AS label,
        at.genre_id,
        at.split,
        at.duration_sec,
        at.sample_rate,
        at.channels
    FROM audio_track at
    JOIN music_genre mg ON at.genre_id = mg.genre_id
    WHERE at.corrupted_flagged = FALSE
    AND   at.duplicate_flagged = FALSE;
"""

with engine.begin() as conn:
    conn.execute(text(CREATE_VIEW))
    print('✓ View vw_clean_tracks created')

✓ View vw_clean_tracks created


In [ ]:
#insert genres into music_genre table
INSERT_GENRES = """
INSERT INTO music_genre (genre_name) VALUES
('blues'),
('classical'),
('country'),
('disco'),
('hiphop'),
('jazz'),
('metal'),
('pop'),
('reggae'),
('rock')
ON CONFLICT (genre_name) DO NOTHING;
"""

with engine.begin() as conn:
    conn.execute(text(INSERT_GENRES))

df_genres = pd.read_sql('SELECT * FROM music_genre ORDER BY genre_name', engine)
print(f'✓ {len(df_genres)} genres inserted')
print(df_genres)

✓ 10 genres inserted
   genre_id genre_name
0         1      blues
1         2  classical
2         3    country
3         4      disco
4         5     hiphop
5         6       jazz
6         7      metal
7         8        pop
8         9     reggae
9        10       rock


In [ ]:
def get_file_hash(file_path):
    """Generate MD5 hash to detect duplicate files"""
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def get_song_id(filename):
    """
    Extract the original song id from a chunk filename.
    Example: 'jazz.00086.03.wav' -> 'jazz.00086'
    """
    parts = filename.split('.')
    return '.'.join(parts[:-2])


genre_map = pd.read_sql(
    'SELECT genre_id, genre_name FROM music_genre', engine
).set_index('genre_name')['genre_id'].to_dict()

tracks = []
print('Scanning chunked dataset...\n')

for genre_name, genre_id in genre_map.items():
    genre_path = GTZAN_PATH / genre_name
    wav_files = sorted(genre_path.glob('*.wav'))

    for wav_file in wav_files:
        try:
            stat = wav_file.stat()
            tracks.append({
                'genre_id':         genre_id,
                'file_path':        str(wav_file),
                'file_name':        wav_file.name,
                'song_id':          get_song_id(wav_file.name),
                'file_size_bytes':  stat.st_size,
                'file_hash':        get_file_hash(wav_file),
                'corrupted_flagged': False,
                'duplicate_flagged': False,
                'split':            None,
                'duration_sec':     None,
                'sample_rate':      None,
                'channels':         None,
            })
        except Exception as e:
            print(f'  ✗ Error: {wav_file.name}: {e}')

print(f'✓ {len(tracks)} chunks found (expect ~9,981, not 1,000)')

# Scanning chunked dataset

In [27]:
def get_file_hash(file_path):
    """Generate MD5 hash to detect duplicate files"""
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def get_song_id(filename):
    """
    Extract the original song id from a chunk filename.
    Example: 'jazz.00086.03.wav' -> 'jazz.00086'
    """
    parts = filename.split('.')
    return '.'.join(parts[:-2])


genre_map = pd.read_sql(
    'SELECT genre_id, genre_name FROM music_genre', engine
).set_index('genre_name')['genre_id'].to_dict()

tracks = []
print('Scanning chunked dataset...\n')

for genre_name, genre_id in genre_map.items():
    genre_path = GTZAN_PATH / genre_name
    wav_files = sorted(genre_path.glob('*.wav'))

    for wav_file in wav_files:
        try:
            stat = wav_file.stat()
            tracks.append({
                'genre_id':         genre_id,
                'file_path':        str(wav_file),
                'file_name':        wav_file.name,
                'song_id':          get_song_id(wav_file.name),
                'file_size_bytes':  stat.st_size,
                'file_hash':        get_file_hash(wav_file),
                'corrupted_flagged': False,
                'duplicate_flagged': False,
                'split':            None,
                'duration_sec':     None,
                'sample_rate':      None,
                'channels':         None,
            })
        except Exception as e:
            print(f'  ✗ Error: {wav_file.name}: {e}')

print(f'✓ {len(tracks)} chunks found (expect ~9,981, not 1,000)')

Scanning chunked dataset...

✓ 9981 chunks found (expect ~9,981, not 1,000)


In [28]:
df_tracks = pd.DataFrame(tracks)
df_tracks.to_sql('audio_track', engine, if_exists='append', index=False)
print(f'✓ {len(df_tracks)} tracks inserted into audio_track')

✓ 9981 tracks inserted into audio_track


# Corrupted chunks flagged 

In [29]:
# Flag ALL chunks that came from the known corrupted source file jazz.00054.wav
# Using song_id instead of the old exact filename, since that file is now split into many chunks
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE audio_track
        SET corrupted_flagged = TRUE
        WHERE song_id = 'jazz.00054'
    """))
    print('✓ Corrupted chunks flagged (all chunks of jazz.00054)')

# Flag duplicate chunks — unchanged, still works correctly per physical file
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE audio_track
        SET duplicate_flagged = TRUE
        WHERE file_hash IN (
            SELECT file_hash
            FROM audio_track
            GROUP BY file_hash
            HAVING COUNT(*) > 1
        )
    """))
    print('✓ Duplicate chunks flagged')

✓ Corrupted chunks flagged (all chunks of jazz.00054)
✓ Duplicate chunks flagged


In [30]:
random.seed(42)

df_clean = pd.read_sql("""
    SELECT track_id, genre_id, song_id
    FROM audio_track
    WHERE corrupted_flagged = FALSE
    AND   duplicate_flagged = FALSE
    ORDER BY genre_id, song_id, track_id
""", engine)

print(f'Clean chunks: {len(df_clean)}')

song_level = df_clean[['genre_id', 'song_id']].drop_duplicates()
print(f'Unique songs (clean): {len(song_level)}')

song_splits = {}
for genre_id, group in song_level.groupby('genre_id'):
    song_ids = group['song_id'].tolist()
    random.shuffle(song_ids)

    n = len(song_ids)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    for i, song_id in enumerate(song_ids):
        if i < n_train:
            song_splits[song_id] = 'train'
        elif i < n_train + n_val:
            song_splits[song_id] = 'val'
        else:
            song_splits[song_id] = 'test'

splits = [
    {'track_id': row['track_id'], 'split': song_splits[row['song_id']]}
    for _, row in df_clean.iterrows()
]

with engine.begin() as conn:
    for row in splits:
        conn.execute(text("""
            UPDATE audio_track
            SET split = :split
            WHERE track_id = :track_id
        """), row)

print('✓ Split assigned (by song, inherited by all its chunks)')

df_split = pd.read_sql("""
    SELECT split, COUNT(*) as count
    FROM audio_track
    WHERE corrupted_flagged = FALSE
    GROUP BY split
    ORDER BY split
""", engine)
print(df_split)

leakage_check = pd.read_sql("""
    SELECT song_id, COUNT(DISTINCT split) as n_splits
    FROM audio_track
    WHERE corrupted_flagged = FALSE AND duplicate_flagged = FALSE
    GROUP BY song_id
    HAVING COUNT(DISTINCT split) > 1
""", engine)

if len(leakage_check) > 0:
    print(f'🚨 LEAKAGE: {len(leakage_check)} songs have chunks in multiple splits!')
else:
    print('✅ No leakage — every song\'s chunks are entirely in one split')

Clean chunks: 9695
Unique songs (clean): 971
✓ Split assigned (by song, inherited by all its chunks)
   split  count
0   test   1530
1  train   6759
2    val   1406
3    NaN    286
✅ No leakage — every song's chunks are entirely in one split


# Updated vw_clean tracks 

In [32]:
CREATE_VIEW = """
CREATE OR REPLACE VIEW vw_clean_tracks AS
    SELECT 
        at.track_id,
        at.file_path,
        mg.genre_name AS label,
        at.genre_id,
        at.split,
        at.duration_sec,
        at.sample_rate,
        at.channels,
        at.song_id
    FROM audio_track at
    JOIN music_genre mg ON at.genre_id = mg.genre_id
    WHERE at.corrupted_flagged = FALSE
    AND   at.duplicate_flagged = FALSE;
"""

with engine.begin() as conn:
    conn.execute(text(CREATE_VIEW))
    print('✓ View vw_clean_tracks updated (now includes song_id)')

✓ View vw_clean_tracks updated (now includes song_id)


SQL workflow fix — steps taken

Branch: fix/split-by-song-not-chunk
Notebooks touched: preprocess_clean_final.ipynb, Sql_Workflow.ipynb

1. Diagnosed the issue

vw_clean_tracks pointed at raw 30-sec files (genres_original/), one row per song (971 total) — not the chunked 3-sec clips the pipeline was designed to use. Root cause: Sql_Workflow.ipynb's GTZAN_PATH never pointed at processed/, and no song_id existed to safely split chunks by song.

2. Regenerated real 3-sec chunks


Fixed broken DATA_ROOT / GENRES_DIR paths in preprocess_clean_final.ipynb.
Re-ran the chunking cell → produced 9,981 chunk files across 10 genres.
Removed 100 stale single-number files per genre left over from earlier failed runs.


3. Rebuilt the database schema


Dropped vw_clean_tracks, audio_track, music_genre.
Recreated audio_track with a new song_id column (groups all chunks of the same song).
Recreated music_genre, the view, and re-inserted the 10 genres.


4. Re-scanned the chunked dataset


Scanned processed/ (not genres_original/) → 9,981 chunks inserted into audio_track, each tagged with its song_id.


5. Fixed corrupted/duplicate flagging


Changed corrupt-file flag from exact filename match to song_id match, so all chunks of jazz.00054 are correctly flagged.
Duplicate flagging (MD5 hash) unchanged — still works per physical chunk.


6. Reassigned the train/val/test split — by song, not by chunk


Split assigned once per unique song (stratified by genre, seed 42).
Every chunk inherits its song's split.
Verified: no song has chunks in more than one split (zero leakage).


Result


9,695 clean chunks (9,981 minus corrupted/duplicated), across 971 unique songs.
Split: 6,759 train / 1,406 val / 1,530 test (~70/14.5/15.8%).
vw_clean_tracks now exposes song_id for downstream notebooks.

# ERD SCHEMA 


-- Table: music_genre
CREATE TABLE music_genre (
    genre_id   SERIAL PRIMARY KEY,
    genre_name VARCHAR(20) NOT NULL UNIQUE
);

-- Table: audio_track
CREATE TABLE audio_track (
    track_id          SERIAL PRIMARY KEY,
    genre_id          INT NOT NULL REFERENCES music_genre(genre_id) ON DELETE CASCADE,
    file_path         TEXT NOT NULL,
    file_name         VARCHAR(64) NOT NULL,
    song_id           VARCHAR(32) NOT NULL,   -- groups all chunks of the same original song
    duration_sec      NUMERIC,
    sample_rate       INT,
    channels          INT,
    file_size_bytes   BIGINT,
    file_hash         CHAR(32),
    duplicate_flagged BOOLEAN DEFAULT FALSE,
    corrupted_flagged BOOLEAN DEFAULT FALSE,
    split             VARCHAR(5),             -- 'train' / 'val' / 'test' / NULL if excluded
    created_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- View: vw_clean_tracks
CREATE OR REPLACE VIEW vw_clean_tracks AS
    SELECT
        at.track_id,
        at.file_path,
        mg.genre_name AS label,
        at.genre_id,
        at.split,
        at.duration_sec,
        at.sample_rate,
        at.channels,
        at.song_id
    FROM audio_track at
    JOIN music_genre mg ON at.genre_id = mg.genre_id
    WHERE at.corrupted_flagged = FALSE
    AND   at.duplicate_flagged = FALSE;